# 07c · 标签：两把刻度，同一把尺子

## 这一章在整个项目里的位置

```
00–06 ✅ → [07 特征和基线] → 08 深度学习 → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 07a 设计 ✅
              ├─ 07b 新数据 MaveDB ✅
              ├─ 07c 标签：两把刻度，同一把尺子   ← 你在这里
              ├─ 07d 结构特征和图：AlphaFold
              ├─ 07e 语言模型特征：ESM-2
              └─ 07f 特征表 + 第一个基线模型 + MLflow
```

## 背景：为什么需要这一章

07b 把 MaveDB 原样装进了 bronze。但每组实验的分数尺子都不一样：有的测细胞生长，有的测荧光，有的数值是负的，有的上万。模型要从很多实验里一起学，先得把它们**换算到同一把尺子上**，这就是**标签**。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 4 步：Databricks 的流水线多了一步 silver_mavedb：分类突变、对到 UniProt 编号
   └─ 第 5 步：dbt 在 gold 里建两张表：每组实验的刻度，每个突变的标签
看结果：这里的输出；Databricks 网页的 Catalog → qafi_dev → silver 和 gold
```

## 这一章做的 6 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 标签是什么 | 温度计的两个固定点 |
| 2 | 给突变分类 | 分拣：错义、无义、同义 |
| 3 | 对到 UniProt 编号 | 两把尺子对齐（05 号学过） |
| 4 | 在 Databricks 上跑 | 分拣线开工 |
| 5 | 检查刻度，换算成标签 | 校准合格才出厂 |
| 6 | 和旧标签对照 | 新旧温度计放在一起读数 |

**花费：0**。**时间**：大约 15 分钟。

---

## 准备

运行下面这一格：连上 Databricks。**不用看代码。**

In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse
from qafi_platform.lakehouse import sql
from qafi_core import labels
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "databricks" / "src"))   # the code the Databricks job runs
from lakehouse import mavedb_variants
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG, JOB = "qafi_dev", "qafi training data"
PAPER = ROOT.parent / "QAFI_CODE_NEW" / "data" / "Dataset_60proteins_features.csv.gz"   # 旧论文的训练集，只当参照答案
print("Databricks:", w.config.host)

---

## 第 1 步 · 标签是什么

### 在干什么
弄清楚怎样把尺子各不相同的实验，换算到同一把尺子上。

### 背景
比喻：**摄氏温度**。用两个固定点定刻度：冰水是 0 度，沸水是 100 度。不管什么温度计，只要对准这两个点，读数就能互相比较。

DMS 实验里，正好有两种突变当固定点：
- **同义突变**：DNA 变了，氨基酸没变，所以和野生型一样。定为 **1**
- **无义突变**：提前出现终止密码子，蛋白被截断，所以失活。定为 **0**

每组实验各自量出这两个点（取中位数），然后：

**标签 =（分数 − 无义突变的中位数）÷（同义突变的中位数 − 无义突变的中位数）**

- 1：和野生型一样
- 0：和截断的蛋白一样
- 大于 1：可能比野生型更活跃

极端值截到 −1 到 3 之间，和 Lafita 2024 一样。

### 用了什么技术，为什么
- **为什么用实验里自带的刻度**：不用人去查、去判断，每组实验自己说了算，可复现
- **为什么用中位数，不用平均数**：个别测坏的数值会把平均数带偏，中位数不受影响
- **稳定性实验另算**：ProteinGym 里的 ΔΔG 稳定性实验没有同义、无义突变，但 ΔΔG 是"和野生型比稳定性变了多少"，所以野生型按定义就是 0，当作 1；最差的 1% 当作 0

### 面试怎么说
> "Each assay is put on one scale with its own internal anchors: the median synonymous variant defines 1 and the median nonsense variant 0, as in Gray et al. and Lafita et al. Medians keep a few bad measurements from moving the anchors."

---

## 第 2 步 · 给突变分类

### 在干什么
从每一行的突变名称，认出它是错义、无义还是同义突变。

### 背景
MaveDB 用一种叫 **HGVS** 的标准写法：
- `p.Leu12Pro`：第 12 位的亮氨酸变成脯氨酸，**错义**
- `p.Lys440Ter`：第 440 位变成终止，**无义**
- `p.Leu12=`：第 12 位没变，**同义**；`p.=` 表示"所有同义突变合在一起"

**一个陷阱**：终止写作 `Ter`，正好也是"一个大写字母加两个小写字母"，和氨基酸缩写一模一样。按顺序判断的话，所有无义突变都会先被当成错义突变。所以要**先判断无义，再判断错义**，单元测试里专门有这一条。

多点突变、插入、缺失这次都不用。

### 执行

In [ ]:
examples = ["p.Leu12Pro", "p.Lys440Ter", "p.Leu12=", "p.=", "ENSP00000494040.1:p.Gly439Asp", "p.[Leu12Pro;Ala13Gly]", "p.Leu12del"]
pd.DataFrame([(h, *mavedb_variants.classify(h)) for h in examples], columns=["HGVS", "类别", "位置", "原氨基酸", "新氨基酸"])

### 结果怎么看
- `p.Lys440Ter` 是 `nonsense`，新氨基酸写成 `*`（终止）
- 带蛋白编号前缀的写法（`ENSP...:`）去掉前缀后照样认得出
- 多点突变、缺失是 `other`，不用

### 面试怎么说
> "HGVS parsing has traps: 'Ter' looks like an amino-acid code, so nonsense has to be matched before missense. Unit tests pin each case."

---

## 第 3 步 · 对到 UniProt 编号

### 在干什么
把每个突变的位置，换成 UniProt 标准序列的编号。

### 背景
05 号已经学过：实验里的位置编号，常常和 UniProt 对不上。比如只测了一个结构域，就从结构域开头数。后面的结构特征用的是 UniProt 编号，对不上就会把特征挂到错的氨基酸上。

MaveDB 的数据更乱：**上传的人编号方式不统一**。按规范，位置应该从实验的目标序列开头数，但有人直接用了全长蛋白的编号（比如 DLG4 的 PDZ3 结构域、阿尔茨海默症的 Aβ42），有的要加上 MaveDB 记录的偏移量。

### 用了什么技术，为什么
**每组实验把几种编号方式都试一遍，选原氨基酸和 UniProt 吻合最多的那一种**：
1. 从目标序列开头数：序列完全一样、或者是 UniProt 的一段，都直接对上；有差别的，用"前后 15 个氨基酸"找唯一位置（05 号的方法）
2. 本来就是 UniProt 编号
3. 加上 MaveDB 记录的偏移量；DNA 目标的偏移量按核苷酸算，要除以 3

吻合率低于 95% 的整组实验不用。DNA 写的目标序列，要先翻译成蛋白：标准遗传密码就 64 个密码子，写进代码里，Databricks 就不用另装 Biopython。

我在 1,117 组人类实验上试过：**95.3% 的位置对上了，其中 99.5% 的原氨基酸和 UniProt 一致**。只用"从目标序列开头数"这一种方式时，分别是 90.4% 和 97.9%。

### 面试怎么说
> "Submitters don't all number variants the same way, so for each score set I try target-relative, full-length and offset numberings and keep the one under which reference residues match UniProt, requiring 95% agreement."

---

## 第 4 步 · 在 Databricks 上跑

### 在干什么
流水线多了一步 `silver_mavedb`：分类、对编号、同一个蛋白突变合并成一行。

### 背景
用 DNA 写的实验里，同一个氨基酸替换会因为不同的密码子出现好几次，比如亮氨酸有 6 个密码子。这些合成一行，分数取中位数。

### 用了什么技术，为什么
**Spark 的 `applyInPandas`**：按"每组实验"分组，每组交给一个普通的 pandas 函数处理。这个函数就是第 2、3 步在你电脑上测过的那个，同一份代码，本地测、云上跑。1,100 组实验，Spark 自动分给多台机器同时做。

### 执行

In [ ]:
run("cd databricks && databricks bundle deploy")
run("cd databricks && databricks bundle run training_data --only silver_mavedb")
lakehouse.latest_run(w, JOB)

### 结果怎么看
`silver_mavedb` 是 `SUCCESS`，大约 1 分钟。

silver 表里有什么：

In [ ]:
sql(w, f'''
SELECT kind, position_mapping, count(*) AS variants, round(avg(CAST(ref_matches_uniprot AS INT)), 4) AS ref_matches
FROM {CATALOG}.silver.mavedb_variants GROUP BY ALL ORDER BY variants DESC''')

### 结果怎么看
- 一共约 **227 万**个蛋白突变：错义约 216 万、无义约 7.7 万、同义约 3.1 万
- `position_mapping` 是选中的编号方式：`identical`、`fragment`、`UniProt numbering`、`MaveDB offset`、`window`，或者 `unmapped`
- `ref_matches` 接近 1：原氨基酸和 UniProt 一致

### 面试怎么说
> "One pandas function per score set, applied with applyInPandas: the same code is unit-tested on the laptop and runs distributed on Databricks."

---

## 第 5 步 · 检查刻度，换算成标签

### 在干什么
用 dbt 在 gold 里建两张表，跑全部测试。

### 背景
比喻：**校准合格的温度计才出厂**。两个固定点量不准，换算出来的标签就不可信。所以每组实验先过一遍检查，不合格的不用，并且写明原因：
- 同义突变至少 1 个，无义突变至少 3 个，错义突变超过 50 个
- 两个固定点要分得开：距离至少是错义突变分数波动范围的一半
- 编号吻合率至少 95%

### 用了什么技术，为什么
- **`dim_label_dataset`**：每组实验一行。两个固定点在哪、分得开不开、编号吻合率多少、用不用（`status`）
- **`fct_variant_label`**：每个错义突变一行，标签就是第 1 步的公式
- **门槛写成 dbt 的变量**（[`dbt_project.yml`](../../dbt/dbt_project.yml) 里的 `vars`）：要调整门槛时只改一处，Git 里也有记录
- **测试**：一个单元测试，用编出来的小例子检查公式和截断；三个自定义检查：
  - 每个数据集里每个突变只有一行
  - 标签所在位置的原氨基酸和 UniProt 一致
  - 同义突变换算后正好是 1、无义突变正好是 0，防止公式写反

### 执行

In [ ]:
os.environ.update(lakehouse.dbt_env(w))
run("cd dbt && uv run dbt build --profiles-dir .")

### 结果怎么看
最后一行 `Done. PASS=53 WARN=0 ERROR=0`：全部表建好，全部测试通过。

每组实验用没用、为什么：

In [ ]:
sql(w, f'''
SELECT source, is_domainome, status, count(*) AS datasets
FROM {CATALOG}.gold.dim_label_dataset GROUP BY ALL ORDER BY source, is_domainome, datasets DESC''')

### 结果怎么看
- **MaveDB，Domainome 以外**：79 组用上。没用上的主要原因：无义突变太少（288 组）、没有同义突变（147 组）、蛋白不在人类参考蛋白组里（51 组，多半是异构体）、两个固定点太近（9 组）
- **Domainome**：500 组用上
- **ProteinGym 稳定性实验**：13 组用上；9 组编号对不上的太多

得到了多少标签：

In [ ]:
sql(w, f'''
SELECT source, is_domainome, count(DISTINCT dataset_id) AS datasets, count(DISTINCT uniprot_id) AS proteins, count(*) AS labels,
       round(percentile(label, 0.5), 2) AS median_label
FROM {CATALOG}.gold.fct_variant_label GROUP BY ALL ORDER BY source, is_domainome''')

### 结果怎么看
- 合计约 **457 个蛋白、107 万个标签**。07a 的时候只有 80 个蛋白
- 标签中位数在 0.8 到 0.9 之间：大部分错义突变影响不大，少数严重破坏功能，和常识一致

### 公司里什么情况怎么做
- **被排除的数据要写明原因**，放在表里：以后有人问"为什么这组实验没用"，查一下就知道，也方便调整规则
- 门槛这种"可以商量的数字"不要散落在代码里，集中放、写说明

### 面试怎么说
> "Datasets whose anchors are missing, too close or badly numbered are excluded, and the reason is a column in a dimension table. The thresholds are dbt variables, and a test checks that the anchors map to exactly 1 and 0."

---

## 第 6 步 · 和旧标签对照

### 在干什么
同一个突变，新算法的标签和旧 QAFI 的标签一不一致。

### 背景
比喻：**新旧两个温度计放在同一杯水里读数**。如果新温度计读数的高低顺序和旧的一样，只是刻度略有差别，说明新的没问题。

In [ ]:
paper = pd.read_csv(PAPER, usecols=["uniprot", "protein", "pos", "first", "second", "score_log_normalized"])
ids = ", ".join(f"'{u}'" for u in paper.uniprot.unique())
new = sql(w, f"SELECT dataset_id, uniprot_id, uniprot_position, ref_aa, alt_aa, label FROM {CATALOG}.gold.fct_variant_label WHERE uniprot_id IN ({ids})")
new = new.astype({"uniprot_position": int, "label": float})
compared = labels.compare_with_paper(new, paper)
print(f"{len(compared)} 组对照，Pearson 中位数 {compared.pearson.median():.2f}")
compared.round(2)

### 结果怎么看
- 大约 45 组对照，**Pearson 中位数约 0.90**：同一个实验的突变，新旧标签的高低顺序基本一样
- 新旧中位数略有差别：两套刻度定得不一样。旧的是人工判断野生型是 0 还是 1，再用最差的 1% 定 0；新的用同义、无义突变
- 表格最下面相关性低（0.2–0.4）的几组，都是**同一个蛋白的不同实验**。比如 HXK4、PTEN、VKOR1 新的这组测的是蛋白丰度，旧论文用的是活性；CXCR4 新的这组测的是抗体结合。测的东西不一样，结果本来就不该一样

### 面试怎么说
> "Against the paper's hand-calibrated labels the rankings agree, with a median r of 0.90. The few low ones turned out to be different assays of the same protein, such as abundance instead of activity."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 07c training labels from synonymous and nonsense anchors") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 为什么 |
|---|---|---|
| 1 标签 | 同义 = 1、无义 = 0，中位数定刻度 | 实验自带的固定点，不用人工判断 |
| 2 分类 | HGVS 解析，先判无义再判错义 | `Ter` 长得像氨基酸 |
| 3 编号 | 三种编号方式，选吻合最多的 | 上传者编号方式不统一 |
| 4 silver | applyInPandas，本地测、云上跑 | 同一份代码，1,100 组并行处理 |
| 5 标签 | 刻度检查 + 换算，原因写进表里 | 校准合格才用；被排除的有据可查 |
| 6 对照 | 和旧标签比，排序一致 | 新标签可信；不一致的是不同实验 |

**一句话**：457 个蛋白、107 万个突变，有了同一把尺子上的标签。每个标签都能追溯到它来自哪组实验、刻度怎么定的。

## 面试题

1. **How do you combine assays that measure different things on different scales?**
   > "Rescale each assay between two internal anchors, synonymous variants for wild-type-like and nonsense variants for loss of function, using medians. Then 1 and 0 mean the same thing in every assay."

2. **How do you handle messy public data?**
   > "Expect it: the classifier has unit tests for every notation I found, numbering is chosen by agreement with the reference sequence, and every dataset that fails a check stays in a table with the reason."

3. **How do you know the new labels are right?**
   > "Tests check that the anchors map to exactly 1 and 0 and that every labelled residue matches UniProt; against the old labels the rankings agree, with a median r of 0.90."

**下一步 07d · 结构特征**：从 AlphaFold 的结构算每个位置的特征，并把每个蛋白存成一张"图"，留给 08 号的图神经网络。